<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/model_experiment_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow dagshub lightgbm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow
import mlflow.sklearn
import dagshub
import itertools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

dagshub.init(repo_owner='icosahedron31', repo_name='Walmart-Sales', mlflow=True)

mlflow.set_experiment('LightGBM_Training')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8f313340-203c-427c-841a-a379bc908cba&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=42dd40f8b032816f75a16ad105a33119b85d87c0793c2358e601ad6652eeb138




Accessing as njvar23

Initialized MLflow to track repo "icosahedron31/Walmart-Sales"

Repository icosahedron31/Walmart-Sales initialized!

<Experiment: artifact_location='mlflow-artifacts:/e94560d312dd4d48b48ae4346f02a19f', creation_time=1783081312202, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783081312202, lifecycle_stage='active', name='LightGBM_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_processed.csv')
val = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/val_processed.csv')

train['Date'] = pd.to_datetime(train['Date'])
val['Date'] = pd.to_datetime(val['Date'])

print(train.shape)
print(val.shape)

Mounted at /content/drive
(332778, 29)
(88792, 29)


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/walmart/Transformers')
from CategoryConverter import CategoryConverter

In [ ]:
# Drop zero-importance weeks_since columns
#cols_to_drop = [
#    'weeks_since_super_bowl',
#    'weeks_since_labor_day',
#    'weeks_since_thanksgiving',
#    'weeks_since_christmas'
#]

#train = train.drop(columns=cols_to_drop)
#val = val.drop(columns=cols_to_drop)

#print(f"Columns after dropping: {train.shape[1]}")

Columns after dropping: 41


In [ ]:
train = train.dropna(subset=['lag_52', 'rolling_mean_52'])
val = val.dropna(subset=['lag_52', 'rolling_mean_52'])

print(train.shape)
print(val.shape)

(173973, 29)
(87110, 29)


In [ ]:
print(train.columns.tolist())

['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size', 'Week', 'Month', 'Year', 'Quarter', 'holiday_name', 'lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4', 'rolling_mean_52']


In [ ]:
feature_cols = [
    'Store', 'Dept', 'Week', 'Month', 'Year', 'Quarter',
    'IsHoliday', 'holiday_name', 'Type', 'Size',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'lag_1', 'lag_2', 'lag_4', 'lag_52',
    'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4', 'rolling_mean_52'
]

categorical_features = ['Store', 'Dept', 'Type', 'holiday_name']

X_train = train[feature_cols]
y_train = train['Weekly_Sales']
is_holiday_train = train['IsHoliday']

X_val = val[feature_cols]
y_val = val['Weekly_Sales']
is_holiday_val = val['IsHoliday']

print(f"Features: {len(feature_cols)}, Train: {X_train.shape}, Val: {X_val.shape}")

Features: 27, Train: (173973, 27), Val: (87110, 27)


In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = is_holiday.map({True: 5, False: 1})
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
with mlflow.start_run(run_name='LightGBM_Cleaning'):
    mlflow.log_param('train_rows', len(X_train))
    mlflow.log_param('val_rows', len(X_val))
    mlflow.log_param('num_features', len(feature_cols))
    mlflow.log_param('cutoff_date', '2012-04-01')
    print("Cleaning run logged")

Cleaning run logged
🏃 View run LightGBM_Cleaning at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1/runs/aaa5fc9edce74a1bb61b7df1dad83c67
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1


In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()

for col in categorical_features:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
for col in categorical_features:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

with mlflow.start_run(run_name='LightGBM_Feature_Selection'):

    baseline_model = lgb.LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        objective='regression_l1',
        random_state=42
    )
    baseline_model.fit(X_train, y_train, categorical_feature=categorical_features)

    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': baseline_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(importance_df)

    val_preds = baseline_model.predict(X_val)
    is_holiday_mask = is_holiday_val.values.astype(bool)

    score_wmae = wmae(y_val, val_preds, is_holiday_val)
    score_mae = np.mean(np.abs(y_val.values - val_preds))
    score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - val_preds[is_holiday_mask]))
    score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - val_preds[~is_holiday_mask]))
    score_rmse = np.sqrt(np.mean((y_val.values - val_preds) ** 2))

    mlflow.log_param('top_5_features', importance_df['feature'].head(5).tolist())
    mlflow.log_param('num_features', len(feature_cols))
    mlflow.log_metric('wmae', score_wmae)
    mlflow.log_metric('mae', score_mae)
    mlflow.log_metric('mae_holiday', score_mae_holiday)
    mlflow.log_metric('mae_non_holiday', score_mae_non_holiday)
    mlflow.log_metric('rmse', score_rmse)

    print(f"\nBaseline WMAE: {score_wmae:.2f} | MAE: {score_mae:.2f} | "
          f"Holiday MAE: {score_mae_holiday:.2f} | Non-holiday MAE: {score_mae_non_holiday:.2f} | RMSE: {score_rmse:.2f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.255685 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4468
[LightGBM] [Info] Number of data points in the train set: 173973, number of used features: 27
[LightGBM] [Info] Start training from score 8049.830078
            feature  importance
1              Dept         905
0             Store         447
22           lag_52         399
19            lag_1         318
2              Week         272
26  rolling_mean_52         108
23   rolling_mean_4         103
20            lag_2          81
16        MarkDown3          72
17        MarkDown4          45
25    rolling_std_4          35
7      holiday_name          34
21            lag_4          31
24  rolling_mean_12          29
10      Temperature          24
11       Fuel_Price          23
15        MarkDown2          20
18        MarkDown5          20
6         IsHoliday          11
14        Mark

In [ ]:
with mlflow.start_run(run_name='LightGBM_GridSearch'):

    param_grid = {
        "learning_rate": [0.05, 0.1],
        "num_leaves": [31, 63, 70],
        "max_depth": [7, 10, 15],
        "subsample": [0.8, 1.0]
    }

    is_holiday_mask = is_holiday_val.values.astype(bool)
    best_score = float('inf')
    best_params = {}
    best_metrics = {}

    for lr, leaves, depth, sub in itertools.product(
        param_grid["learning_rate"], param_grid["num_leaves"],
        param_grid["max_depth"], param_grid["subsample"]
    ):
        m = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=lr, num_leaves=leaves,
            max_depth=depth, subsample=sub, objective='regression_l1', random_state=42
        )
        m.fit(
            X_train, y_train, categorical_feature=categorical_features,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)]
        )
        preds = m.predict(X_val)

        score_wmae = wmae(y_val, preds, is_holiday_val)
        score_mae = np.mean(np.abs(y_val.values - preds))
        score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - preds[is_holiday_mask]))
        score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - preds[~is_holiday_mask]))
        score_rmse = np.sqrt(np.mean((y_val.values - preds) ** 2))

        if score_wmae < best_score:
            best_score = score_wmae
            best_params = {'learning_rate': lr, 'num_leaves': leaves, 'max_depth': depth, 'subsample': sub}
            best_metrics = {
                'wmae': score_wmae, 'mae': score_mae,
                'mae_holiday': score_mae_holiday, 'mae_non_holiday': score_mae_non_holiday,
                'rmse': score_rmse, 'best_iteration': m.best_iteration_
            }

        print(f"lr={lr}, leaves={leaves}, depth={depth}, sub={sub} → "
              f"WMAE: {score_wmae:.2f} | Holiday MAE: {score_mae_holiday:.2f}")

    mlflow.log_params(best_params)
    mlflow.log_metrics(best_metrics)
    print(f"\nBest params: {best_params}")
    print(f"Best metrics: {best_metrics}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088087 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4468
[LightGBM] [Info] Number of data points in the train set: 173973, number of used features: 27
[LightGBM] [Info] Start training from score 8049.830078
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[922]	valid_0's l1: 1358.25
lr=0.05, leaves=31, depth=7, sub=0.8 → WMAE: 1370.64 | Holiday MAE: 1463.40
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.064078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4468
[LightGBM] [Info] Number of data points in the train set: 173973, number of used features: 27
[LightGBM] [Info] Start training from score 8049.830078
Training until validation scores don't impro

In [ ]:
with mlflow.start_run(run_name='LightGBM_BestParams_Training'):

    model_best = lgb.LGBMRegressor(
        n_estimators=3000,      # extended cap — grid search's winner hit 999/1000 without early stopping
        learning_rate=0.05,
        num_leaves=31,
        max_depth=10,
        subsample=0.8,
        objective='regression_l1',
        random_state=42
    )

    model_best.fit(
        X_train, y_train,
        categorical_feature=categorical_features,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
    )

    val_preds = model_best.predict(X_val)
    is_holiday_mask = is_holiday_val.values.astype(bool)

    score_wmae = wmae(y_val, val_preds, is_holiday_val)
    score_mae = np.mean(np.abs(y_val.values - val_preds))
    score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - val_preds[is_holiday_mask]))
    score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - val_preds[~is_holiday_mask]))
    score_rmse = np.sqrt(np.mean((y_val.values - val_preds) ** 2))

    mlflow.log_params({
        'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 10,
        'subsample': 0.8, 'n_estimators_cap': 3000, 'objective': 'regression_l1'
    })
    mlflow.log_metric('wmae', score_wmae)
    mlflow.log_metric('mae', score_mae)
    mlflow.log_metric('mae_holiday', score_mae_holiday)
    mlflow.log_metric('mae_non_holiday', score_mae_non_holiday)
    mlflow.log_metric('rmse', score_rmse)
    mlflow.log_metric('best_iteration', model_best.best_iteration_)

    print(f"WMAE: {score_wmae:.2f} | Holiday MAE: {score_mae_holiday:.2f} | Best iteration: {model_best.best_iteration_}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.166991 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4468
[LightGBM] [Info] Number of data points in the train set: 173973, number of used features: 27
[LightGBM] [Info] Start training from score 8049.830078
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 1383.47
[400]	valid_0's l1: 1368.73
[600]	valid_0's l1: 1361.96
[800]	valid_0's l1: 1357.93
[1000]	valid_0's l1: 1353.96
[1200]	valid_0's l1: 1352.18
[1400]	valid_0's l1: 1351.07
Early stopping, best iteration is:
[1393]	valid_0's l1: 1350.46
WMAE: 1360.32 | Holiday MAE: 1434.17 | Best iteration: 1393
🏃 View run LightGBM_BestParams_Training at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1/runs/cb06f1e7ffe7454d809f6b601f14a680
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

train_sorted = train.sort_values('Date').reset_index(drop=True)
X_train_cv = train_sorted[feature_cols].copy()
y_train_cv = train_sorted['Weekly_Sales']
is_holiday_cv_full = train_sorted['IsHoliday']

for col in categorical_features:
    X_train_cv[col] = X_train_cv[col].astype('category')

tscv = TimeSeriesSplit(n_splits=5)
fold_wmae, fold_mae, fold_mae_holiday, fold_mae_non_holiday, fold_rmse = [], [], [], [], []

with mlflow.start_run(run_name='LightGBM_CrossValidation'):
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_cv)):
        X_cv_train, X_cv_val = X_train_cv.iloc[train_idx], X_train_cv.iloc[val_idx]
        y_cv_train, y_cv_val = y_train_cv.iloc[train_idx], y_train_cv.iloc[val_idx]
        is_holiday_cv = is_holiday_cv_full.iloc[val_idx]
        holiday_mask = is_holiday_cv.values.astype(bool)

        m = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.05, num_leaves=31,
            max_depth=10, subsample=0.8, objective='regression_l1', random_state=42
        )
        m.fit(
            X_cv_train, y_cv_train, categorical_feature=categorical_features,
            eval_set=[(X_cv_val, y_cv_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        preds = m.predict(X_cv_val)

        score_wmae = wmae(y_cv_val, preds, is_holiday_cv)
        score_mae = np.mean(np.abs(y_cv_val.values - preds))
        score_mae_holiday = np.mean(np.abs(y_cv_val.values[holiday_mask] - preds[holiday_mask])) if holiday_mask.sum() > 0 else np.nan
        score_mae_non_holiday = np.mean(np.abs(y_cv_val.values[~holiday_mask] - preds[~holiday_mask]))
        score_rmse = np.sqrt(np.mean((y_cv_val.values - preds) ** 2))

        fold_wmae.append(score_wmae); fold_mae.append(score_mae)
        fold_mae_holiday.append(score_mae_holiday); fold_mae_non_holiday.append(score_mae_non_holiday)
        fold_rmse.append(score_rmse)

        print(f"Fold {fold+1}: holiday_weeks={holiday_mask.sum()} → "
              f"WMAE: {score_wmae:.2f} | Holiday MAE: {score_mae_holiday:.2f} | RMSE: {score_rmse:.2f}")

    mlflow.log_metric('cv_mean_wmae', np.nanmean(fold_wmae))
    mlflow.log_metric('cv_std_wmae', np.nanstd(fold_wmae))
    mlflow.log_metric('cv_mean_mae', np.nanmean(fold_mae))
    mlflow.log_metric('cv_mean_mae_holiday', np.nanmean(fold_mae_holiday))
    mlflow.log_metric('cv_mean_mae_non_holiday', np.nanmean(fold_mae_non_holiday))
    mlflow.log_metric('cv_mean_rmse', np.nanmean(fold_rmse))
    mlflow.log_param('n_splits', 5)

    print(f"\nCV Mean WMAE: {np.nanmean(fold_wmae):.2f} ± {np.nanstd(fold_wmae):.2f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010482 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2802
[LightGBM] [Info] Number of data points in the train set: 28998, number of used features: 21
[LightGBM] [Info] Start training from score 8311.974609
Fold 1: holiday_weeks=0 → WMAE: 1510.77 | Holiday MAE: nan | RMSE: 3573.03
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006319 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2983
[LightGBM] [Info] Number of data points in the train set: 57993, number of used features: 21
[LightGBM] [Info] Start training from score 8255.870117
Fold 2: holiday_weeks=0 → WMAE: 1319.18 | Holiday MAE: nan | RMSE: 2813.14
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005546 s

In [ ]:
lgbm_pipeline = Pipeline([
    ('category_converter', CategoryConverter(categorical_features)),
    ('model', lgb.LGBMRegressor(
        n_estimators=1393,   # from BestParams_Training's best_iteration
        learning_rate=0.05,
        num_leaves=31,
        max_depth=10,
        subsample=0.8,
        objective='regression_l1',
        random_state=42
    ))
])

with mlflow.start_run(run_name='LightGBM_Pipeline_Final'):
    lgbm_pipeline.fit(X_train, y_train, model__categorical_feature=categorical_features)

    val_preds_final = lgbm_pipeline.predict(X_val)
    is_holiday_mask = is_holiday_val.values.astype(bool)

    score_wmae = wmae(y_val, val_preds_final, is_holiday_val)
    score_mae = np.mean(np.abs(y_val.values - val_preds_final))
    score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - val_preds_final[is_holiday_mask]))
    score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - val_preds_final[~is_holiday_mask]))
    score_rmse = np.sqrt(np.mean((y_val.values - val_preds_final) ** 2))

    mlflow.log_params({
        'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 10,
        'subsample': 0.8, 'objective': 'regression_l1', 'n_estimators': 1393
    })
    mlflow.log_metric('wmae', score_wmae)
    mlflow.log_metric('mae', score_mae)
    mlflow.log_metric('mae_holiday', score_mae_holiday)
    mlflow.log_metric('mae_non_holiday', score_mae_non_holiday)
    mlflow.log_metric('rmse', score_rmse)

    mlflow.sklearn.log_model(
        lgbm_pipeline,
        'lgbm_pipeline',
        registered_model_name='LightGBM_Best',
        skops_trusted_types=[
            'CategoryConverter.CategoryConverter',
            'collections.OrderedDict',
            'lightgbm.basic.Booster',
            'lightgbm.sklearn.LGBMRegressor'
        ]
    )
    print(f"Pipeline WMAE: {score_wmae:.2f} | Holiday MAE: {score_mae_holiday:.2f} | RMSE: {score_rmse:.2f}")
    print("Pipeline saved and registered as LightGBM_Best")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4468
[LightGBM] [Info] Number of data points in the train set: 173973, number of used features: 27
[LightGBM] [Info] Start training from score 8049.830078


2026/07/11 15:36:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'LightGBM_Best' already exists. Creating a new version of this model...
2026/07/11 15:36:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_Best, version 2
Created version '2' of model 'LightGBM_Best'.


Pipeline WMAE: 1360.32 | Holiday MAE: 1434.17 | RMSE: 3040.30
Pipeline saved and registered as LightGBM_Best
🏃 View run LightGBM_Pipeline_Final at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1/runs/6009f5d509ca47c4b65fc7e5d0d7b532
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/1
